# IVDRS & SUDORS community: records spreadsheet

Read-only: asks Prism for every published record in the community and writes `IVDRS_SUDORS_Prism_Records.xlsx`
with one tab per proposed child community (IVDRS Reports and Articles, SUDORS Reports and Articles,
Peer-reviewed Publications) plus an "All records" tab. Each row has the **DOI to cite**: the parent DOI for reports
(it never changes when a new version is added) and the publisher's DOI for journal articles.
Keep `prism_deposit.py` in the same folder.

In [1]:
from pathlib import Path
import importlib
import prism_deposit as pd_
pd_ = importlib.reload(pd_)

ENV        = "production"                       # or "staging"
PRISM_URL  = "https://prism.northwestern.edu"   # staging: https://prism-staging.fsm.northwestern.edu
TOKEN_VAR  = "PRISM_PRODUCTION_TOKEN"           # staging: PRISM_STAGING_TOKEN
CONFIG_FILE = Path("config.py")
COMMUNITY_SLUG = "ivdrs-sudors"
OUTPUT_DIR = Path("output")
OUT_XLSX   = OUTPUT_DIR / "IVDRS_SUDORS_Prism_Records.xlsx"

env = pd_.load_config(CONFIG_FILE)
print(f"{TOKEN_VAR} configured: {bool(env.get(TOKEN_VAR))}")
api = pd_.Prism(PRISM_URL, env.get(TOKEN_VAR, ""), verify=True)
reg = pd_.Registry(OUTPUT_DIR / f"{ENV}_registry.json")     # used only to add record_id to each row

PRISM_PRODUCTION_TOKEN configured: True


In [2]:
community = api.get(f"/api/communities/{COMMUNITY_SLUG}")
assert community, f"No community '{COMMUNITY_SLUG}' on {PRISM_URL}"
records = pd_.community_records(api, community["id"])
rows = pd_.report_rows(records, reg)
counts = pd_.save_report(rows, OUT_XLSX, community["metadata"]["title"])
print(counts)
print("Saved", OUT_XLSX.resolve())
missing = [r["Title"] for r in rows if not r["DOI to cite"]]
no_parent = [r["record_id"] or r["Title"] for r in rows if not r["Parent DOI"] and not r["_external"]]
if missing: print("No DOI at all:", missing)
if no_parent: print("Reports without a parent DOI (version DOI used):", no_parent)

86 published records in the community
{'All records': 86, 'IVDRS Reports and Articles': 46, 'SUDORS Reports and Articles': 21, 'Peer-reviewed Publications': 39}
Saved C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\output\IVDRS_SUDORS_Prism_Records.xlsx
